# Lab 01 — Spark Architecture, Execution Model & Partitions

**Exam domain:** Apache Spark Architecture and Components (20% of the exam)

**Runs on:** Databricks Free Edition (serverless notebook) or any local PySpark 3.5 install.

**Time:** ~90 minutes

Coming from Azure Synapse you already *use* Spark's execution model every day — but the exam asks you to
*name* things: job vs stage vs task vs slot, narrow vs wide, storage levels, what the driver does versus
what an executor does. This lab makes those names concrete.

> **Note on Databricks Free Edition:** it is serverless-only, so `sc` internals like
> `spark.sparkContext.defaultParallelism` and some `spark.conf` writes may be restricted. Every cell below
> is wrapped so it degrades gracefully — if a cell errors on serverless, read the markdown and move on, or
> run the lab against local PySpark where you have full control.

In [ ]:
# --- Setup: works both on Databricks and on plain PySpark ---
try:
    spark  # Databricks provides this
    print("Using the Databricks-provided SparkSession")
except NameError:
    from pyspark.sql import SparkSession
    spark = (SparkSession.builder
             .appName("lab01-architecture")
             .master("local[4]")            # 4 executor slots on the local machine
             .config("spark.sql.shuffle.partitions", "8")
             .getOrCreate())
    print("Created a local SparkSession")

print("Spark version:", spark.version)

## 1. The execution hierarchy — the vocabulary the exam tests

Memorise this chain. Exam questions are frequently just "which of these is the smallest unit of work?"

| Term | What it is | Rule of thumb |
|---|---|---|
| **Application** | One `SparkSession` / one driver program | 1 app = 1 driver |
| **Job** | Everything triggered by **one action** | `count()` → 1 job (sometimes more) |
| **Stage** | A set of tasks with **no shuffle between them** | new stage ⇔ a shuffle boundary |
| **Task** | Work on **one partition** | #tasks in a stage = #partitions |
| **Slot** | One CPU core on an executor; runs one task at a time | parallelism = total slots |

**Driver** — runs `main()`, builds the logical plan, asks the cluster manager for executors, schedules tasks,
collects results. Single point of failure. `collect()` pulls data *into the driver* — this is the classic
OOM question on the exam.

**Executor** — a JVM process on a worker node. Runs tasks, holds cached partitions in its own memory,
reports back to the driver. Executors do **not** talk to each other except during shuffles.

**Worker node** — the machine. It can host one or more executors.

**Cluster manager** — YARN / Kubernetes / Standalone / (on Databricks, managed for you). Allocates the
containers the executors run in.

In [ ]:
# Inspect the pieces of the runtime you are allowed to see
def safe(label, fn):
    try:
        print(f"{label:38}: {fn()}")
    except Exception as e:
        print(f"{label:38}: <not available here> ({type(e).__name__})")

safe("App name",              lambda: spark.conf.get("spark.app.name"))
safe("Default shuffle partitions", lambda: spark.conf.get("spark.sql.shuffle.partitions"))
safe("AQE enabled",           lambda: spark.conf.get("spark.sql.adaptive.enabled"))
safe("AQE coalesce partitions",lambda: spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
safe("AQE skew join",         lambda: spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
safe("Broadcast threshold (bytes)", lambda: spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
safe("defaultParallelism",    lambda: spark.sparkContext.defaultParallelism)

## 2. Transformations vs actions, and lazy evaluation

- **Transformation** → returns a new DataFrame, **does nothing yet**. `select`, `filter`, `withColumn`,
  `join`, `groupBy().agg()`, `repartition`, `orderBy`, `distinct`, `union`.
- **Action** → triggers a job and returns a value (or writes). `count`, `collect`, `take`, `first`, `show`,
  `write.save`, `foreach`, `toPandas`, `head`.

Two flavours of transformation:

- **Narrow** — each output partition depends on **one** input partition. No data moves across the network.
  `select`, `filter`, `withColumn`, `drop`, `union`, `coalesce`, `mapPartitions`.
- **Wide** — output partitions depend on **many** input partitions ⇒ a **shuffle** ⇒ a new **stage**.
  `groupBy`, `join` (non-broadcast), `distinct`, `dropDuplicates`, `orderBy`/`sort`, `repartition`,
  window functions with a `partitionBy`.

⚠️ Exam traps: `coalesce()` is **narrow** (it merges partitions, no full shuffle). `repartition()` is
**wide**. `union` is narrow. `cache()` is neither a transformation nor an action — it's lazy, and only
materialises when the next action runs.

In [ ]:
from pyspark.sql import functions as F

df = spark.range(0, 1_000_000).withColumn("bucket", (F.col("id") % 7).cast("int"))

t0 = df.filter(F.col("bucket") > 2).select("id", "bucket")   # narrow, lazy — no job runs
print("Nothing has executed yet. Type of t0:", type(t0).__name__)

t1 = t0.groupBy("bucket").count()                            # wide, still lazy
print("Still nothing. Now the action:")
t1.show()                                                    # <- job runs here

In [ ]:
# Read the plan. `explain(True)` shows parsed -> analyzed -> optimized -> physical.
t1.explain(True)

In [ ]:
# `explain("formatted")` is the friendliest form. Look for:
#   Exchange           = a shuffle (stage boundary)
#   AdaptiveSparkPlan  = AQE is on
#   PhotonX / *(n)     = whole-stage codegen groupings
t1.explain("formatted")

### ✍️ Exercise 2.1
For each expression below, say (a) narrow or wide, (b) how many jobs the whole cell triggers.

```python
a = df.withColumn("x", F.col("id") * 2)
b = a.repartition(10)
c = b.filter("x > 100")
d = c.orderBy("x")
print(d.count())
print(d.take(3))
```

<details><summary>Answer</summary>

`withColumn` narrow · `repartition` **wide** · `filter` narrow · `orderBy` **wide**.
Two actions (`count`, `take`) ⇒ at least **2 jobs**. `orderBy` also runs a sampling job to compute range
boundaries, so you will usually see **3+** jobs in the Spark UI. Nothing is cached, so the whole chain is
recomputed for the second action — this is the point of the next section.
</details>

## 3. Partitions — the unit of parallelism

Three different numbers, and the exam expects you to keep them apart:

1. **Input partitions** — determined at read time by file sizes and
   `spark.sql.files.maxPartitionBytes` (default 128 MB).
2. **Shuffle partitions** — `spark.sql.shuffle.partitions`, default **200**. This is the number of
   partitions *after* a wide transformation. The single most commonly tuned setting.
3. **Output partitions** — one file per partition when you write, unless `partitionBy` splits further.

| Operation | Shuffle? | Can increase? | Can decrease? | Notes |
|---|---|---|---|---|
| `repartition(n)` | full shuffle | ✅ | ✅ | even distribution; also `repartition(n, "col")` |
| `repartitionByRange(n, "col")` | shuffle (with sampling) | ✅ | ✅ | ordered ranges, good before range writes |
| `coalesce(n)` | **no** shuffle | ❌ | ✅ | merges local partitions; can cause skew |

In [ ]:
print("partitions after range()      :", df.rdd.getNumPartitions())
print("partitions after repartition(12):", df.repartition(12).rdd.getNumPartitions())
print("partitions after coalesce(2)    :", df.repartition(12).coalesce(2).rdd.getNumPartitions())

# coalesce can NOT increase the count — this silently stays at 4-ish, it does not become 50
print("coalesce(50) on a small df      :", df.coalesce(50).rdd.getNumPartitions())

In [ ]:
# How rows are actually spread across partitions
from pyspark.sql.functions import spark_partition_id

(df.repartition(6, "bucket")
   .withColumn("pid", spark_partition_id())
   .groupBy("pid").count()
   .orderBy("pid")
   .show())
# Note: 7 distinct buckets hashed into 6 partitions => uneven. This is hash skew in miniature.

### ✍️ Exercise 3.1
You read 40 GB of Parquet and immediately `groupBy("customer_id").sum("amount")`, then write the result.
Default settings, AQE off.

1. Roughly how many input partitions?
2. How many partitions after the groupBy?
3. How many files does the write produce?

<details><summary>Answer</summary>

1. 40 GB ÷ 128 MB ≈ **320** input partitions.
2. **200** — `spark.sql.shuffle.partitions` default, regardless of input size.
3. **200** files (one per partition), many of them tiny. With AQE's `coalescePartitions` enabled Spark
   would merge them down at runtime; without AQE you'd `coalesce()` or `repartition()` before writing.
</details>

## 4. Caching and storage levels

`cache()` == `persist(StorageLevel.MEMORY_AND_DISK)` for **DataFrames** (note: for RDDs the default is
`MEMORY_ONLY` — a favourite exam distinction).

| Storage level | Memory | Disk | Serialized | Replicas |
|---|---|---|---|---|
| `MEMORY_ONLY` | ✅ | ❌ | ❌ (Python: always serialized) | 1 |
| `MEMORY_AND_DISK` | ✅ | ✅ spill | ❌ | 1 |
| `MEMORY_ONLY_SER` | ✅ | ❌ | ✅ | 1 |
| `MEMORY_AND_DISK_SER` | ✅ | ✅ | ✅ | 1 |
| `DISK_ONLY` | ❌ | ✅ | ✅ | 1 |
| `*_2` variants | | | | 2 |
| `OFF_HEAP` | off-heap | ❌ | ✅ | 1 |

Key facts:
- `cache()`/`persist()` are **lazy** — nothing is stored until an action materialises the DataFrame.
- `unpersist()` is **eager** and blocking-optional: `unpersist(blocking=False)` by default.
- Cache when a DataFrame is reused **more than once** and recomputation is expensive. Caching something
  used once is a net loss (memory pressure + eviction + GC).
- Eviction is **LRU**. A `MEMORY_ONLY` partition that gets evicted is **recomputed**, not spilled.

In [ ]:
from pyspark import StorageLevel
import time

wide = spark.range(0, 3_000_000).withColumn("v", F.rand())
agg  = wide.groupBy((F.col("id") % 1000).alias("k")).agg(F.avg("v").alias("avg_v"))

t = time.time(); agg.count(); cold = time.time() - t

agg.persist(StorageLevel.MEMORY_AND_DISK)
agg.count()                                  # materialises the cache
t = time.time(); agg.count(); warm = time.time() - t

print(f"uncached: {cold:.2f}s   cached: {warm:.2f}s")
print("storage level:", agg.storageLevel)
agg.unpersist()

### ✍️ Exercise 4.1
Which of these is *not* a good reason to cache?

- A. A DataFrame used as the left side of three different joins.
- B. A DataFrame you filter once and write straight to Parquet.
- C. An expensive parsed-JSON DataFrame explored interactively in a notebook.
- D. A lookup table reused across many iterations of an ML loop.

<details><summary>Answer</summary>
**B** — single use, single action. Caching adds memory pressure with zero reuse benefit.
</details>

## 5. Spark modules — know what lives where

- **Spark Core** — RDDs, scheduling, memory management, fault recovery. Everything sits on top of it.
- **Spark SQL** — Catalyst optimizer, the DataFrame/Dataset API, the SQL parser, Tungsten execution.
  In Python `DataFrame` **is** `Dataset[Row]`; the typed `Dataset` API is Scala/Java only.
- **Structured Streaming** — the streaming DataFrame API, built on Spark SQL.
- **MLlib** — DataFrame-based ML (`pyspark.ml`); the older RDD-based `pyspark.mllib` is in maintenance.
- **GraphX** — RDD-based graph processing, **Scala/Java only** (no PySpark API).
- **Pandas API on Spark** — `pyspark.pandas`, a pandas-compatible surface over Spark SQL.

### Advantages / challenges of Spark (a real exam sub-objective)
**Advantages:** in-memory computation, unified batch + streaming + SQL + ML, lazy optimisation via Catalyst,
fault tolerance via lineage, language choice, scales horizontally.
**Challenges:** no built-in storage layer, expensive shuffles, sensitive to data skew and small files,
JVM garbage-collection tuning, Python UDF serialization cost, not designed for low-latency
single-row lookups (that's an OLTP database's job).

## 6. Read the Spark UI

Open the Spark UI (Databricks: **View → Spark UI** on the cell, or the cluster's Spark UI tab).
Run the cell below, then find:

1. **Jobs** tab — how many jobs did the cell create?
2. Click a job → **Stages**. Count the stages. Each `Exchange` in the plan = one boundary.
3. Open the widest stage → the **Summary Metrics** table. Compare *Min / Median / Max* for
   **Duration**, **Shuffle Read Size** and **Spill**. A Max far above the Median = **skew**.
4. **SQL / DataFrame** tab → click the query → look at the graph. Hover over `Exchange` to see bytes shuffled.
5. **Storage** tab — your cached DataFrame appears here with its fraction cached.

In [ ]:
left  = spark.range(0, 2_000_000).withColumn("k", (F.col("id") % 5000).cast("int"))
right = spark.range(0, 5000).withColumnRenamed("id", "k").withColumn("label", F.concat(F.lit("k_"), F.col("k")))

joined = left.join(right, on="k", how="inner").groupBy("label").count()
print("rows:", joined.count())
joined.explain("formatted")

## 7. Wrap-up checklist

Tick these off before you move to Lab 02. If you can't answer one without scrolling up, redo that section.

- [ ] I can order: application → job → stage → task, and say what creates each boundary.
- [ ] I can list 5 narrow and 5 wide transformations without hesitating.
- [ ] I know the default of `spark.sql.shuffle.partitions` (200) and of `maxPartitionBytes` (128 MB).
- [ ] I know `coalesce` cannot increase partitions and does not shuffle.
- [ ] I can name every storage level and say which one `cache()` uses for a DataFrame.
- [ ] I know `cache()` is lazy and `unpersist()` is eager.
- [ ] I can point at `Exchange` in a plan and say "that's a shuffle, that's a stage boundary".
- [ ] I know what the driver does that an executor doesn't (and why `collect()` is dangerous).